# Song Recommendations (OpenAI)

Enter your **mood** and **time of day** to get personalized song picks powered by OpenAI.

**Setup:** Run the install cell once, then run the rest top to bottom. Your API key is loaded from `server/.env`.

In [ ]:
%pip install -q openai python-dotenv pandas

: 

In [ ]:
import json
import os
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# Load OPENAI_API_KEY from server/.env
env_path = Path("server/.env")
if not env_path.exists():
    raise FileNotFoundError(f"Could not find {env_path.resolve()}. Run this notebook from the PlaylisterYT folder.")

load_dotenv(env_path)
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is missing from server/.env")

client = OpenAI(api_key=api_key)
print("OpenAI client ready.")

: 

In [ ]:
def get_time_of_day_label(hour: int) -> str:
    if 5 <= hour < 12:
        return "morning"
    if 12 <= hour < 17:
        return "afternoon"
    if 17 <= hour < 21:
        return "evening"
    return "night"


def recommend_songs(mood: str, time_of_day: str | None = None, genre: str | None = None) -> list[dict]:
    mood = mood.strip()
    if not mood:
        raise ValueError("Mood is required.")

    detected_time = time_of_day.strip() if time_of_day and time_of_day.strip() else get_time_of_day_label(datetime.now().hour)
    genre_line = f"Preferred genre: {genre.strip()}." if genre and genre.strip() else ""

    user_prompt = f"""Recommend exactly 6 real, well-known songs for:
- Mood: {mood}
- Time of day: {detected_time}
{genre_line}

Include a diverse mix of eras. For each song output ONLY this JSON (no markdown, no extra text):
[{{"title":"...","artist":"...","year":2020,"genre":"...","reason":"one sentence why it fits"}}]"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a music recommendation API. Respond ONLY with a valid JSON array. "
                    "No markdown, no code fences, no explanation — raw JSON only."
                ),
            },
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.7,
        max_tokens=1024,
    )

    raw_text = (response.choices[0].message.content or "").strip()
    match = re.search(r"\[[\s\S]*\]", raw_text)
    if not match:
        raise ValueError(f"Model did not return a JSON array:\n{raw_text}")

    return json.loads(match.group(0))

In [ ]:
# --- Your inputs ---
mood = input("Mood or vibe (e.g. Chill, Focus, Party): ")
time_of_day = input("Time of day (morning/afternoon/evening/night, or press Enter for auto-detect): ")
genre = input("Optional genre (press Enter to skip): ")

print(f"\nUsing mood='{mood}' | time='{time_of_day or 'auto'}' | genre='{genre or 'any'}'")

In [ ]:
recommendations = recommend_songs(mood=mood, time_of_day=time_of_day, genre=genre)

df = pd.DataFrame(recommendations)
display(df[["title", "artist", "year", "genre", "reason"]])

print(f"\n{len(recommendations)} songs recommended for '{mood.strip()}'")